In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
emp_data = [
(101,"Amit",25,"IT","Pune",40000,2),
(102,"Neha",30,"HR","Mumbai",50000,5),
(103,"Rahul",28,"IT","Pune",45000,3),
(104,"Sneha",35,"Finance","Delhi",60000,8),
(105,"Raj",26,"IT","Mumbai",42000,2),
(106,"Pooja",32,"HR","Pune",52000,6),
(107,"Karan",29,"Finance","Mumbai",58000,7),
(108,"Meena",27,"IT","Delhi",46000,3)
]
column=["emp_id", "name", "age", "department", "city", "salary", "experience"]
df = spark.createDataFrame(emp_data, column)
df.show()

+------+-----+---+----------+------+------+----------+
|emp_id| name|age|department|  city|salary|experience|
+------+-----+---+----------+------+------+----------+
|   101| Amit| 25|        IT|  Pune| 40000|         2|
|   102| Neha| 30|        HR|Mumbai| 50000|         5|
|   103|Rahul| 28|        IT|  Pune| 45000|         3|
|   104|Sneha| 35|   Finance| Delhi| 60000|         8|
|   105|  Raj| 26|        IT|Mumbai| 42000|         2|
|   106|Pooja| 32|        HR|  Pune| 52000|         6|
|   107|Karan| 29|   Finance|Mumbai| 58000|         7|
|   108|Meena| 27|        IT| Delhi| 46000|         3|
+------+-----+---+----------+------+------+----------+



In [0]:
dept_data = [
("IT","Rajesh","Pune"),
("HR","Sunita","Mumbai"),
("Finance","Anil","Delhi")
]
column=["department", "manager", "manager_city"]
df1= spark.createDataFrame(dept_data,column)
df1.show()

+----------+-------+------------+
|department|manager|manager_city|
+----------+-------+------------+
|        IT| Rajesh|        Pune|
|        HR| Sunita|      Mumbai|
|   Finance|   Anil|       Delhi|
+----------+-------+------------+



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col
from pyspark.sql.functions import rank
window = Window.partitionBy("department").orderBy(col("salary").desc())
df.withColumn("dense_rank",dense_rank().over(window)).limit(3).show()

+------+-----+---+----------+------+------+----------+----------+
|emp_id| name|age|department|  city|salary|experience|dense_rank|
+------+-----+---+----------+------+------+----------+----------+
|   104|Sneha| 35|   Finance| Delhi| 60000|         8|         1|
|   107|Karan| 29|   Finance|Mumbai| 58000|         7|         2|
|   106|Pooja| 32|        HR|  Pune| 52000|         6|         1|
+------+-----+---+----------+------+------+----------+----------+



In [0]:
window = Window.partitionBy("department").orderBy(col("salary").desc())
df.withColumn("row_number", row_number().over(window)).limit(1).show()

+------+-----+---+----------+-----+------+----------+----------+
|emp_id| name|age|department| city|salary|experience|row_number|
+------+-----+---+----------+-----+------+----------+----------+
|   104|Sneha| 35|   Finance|Delhi| 60000|         8|         1|
+------+-----+---+----------+-----+------+----------+----------+



In [0]:
df.groupBy("department").sum("salary").show()

+----------+-----------+
|department|sum(salary)|
+----------+-----------+
|        IT|     173000|
|        HR|     102000|
|   Finance|     118000|
+----------+-----------+



In [0]:
windowSpec = Window.partitionBy("department").orderBy("salary").rowsBetween(Window.unboundedPreceding, Window.currentRow)
df.withColumn("running_salary",sum("salary").over(windowSpec)).show()

+------+-----+---+----------+------+------+----------+--------------+
|emp_id| name|age|department|  city|salary|experience|running_salary|
+------+-----+---+----------+------+------+----------+--------------+
|   107|Karan| 29|   Finance|Mumbai| 58000|         7|         58000|
|   104|Sneha| 35|   Finance| Delhi| 60000|         8|        118000|
|   102| Neha| 30|        HR|Mumbai| 50000|         5|         50000|
|   106|Pooja| 32|        HR|  Pune| 52000|         6|        102000|
|   101| Amit| 25|        IT|  Pune| 40000|         2|         40000|
|   105|  Raj| 26|        IT|Mumbai| 42000|         2|         82000|
|   103|Rahul| 28|        IT|  Pune| 45000|         3|        127000|
|   108|Meena| 27|        IT| Delhi| 46000|         3|        173000|
+------+-----+---+----------+------+------+----------+--------------+



In [0]:
windowSpec = Window.partitionBy("department")

df.withColumn("avg_salary",avg("salary").over(windowSpec)).filter("salary > avg_salary").show()

+------+-----+---+----------+-----+------+----------+----------+
|emp_id| name|age|department| city|salary|experience|avg_salary|
+------+-----+---+----------+-----+------+----------+----------+
|   103|Rahul| 28|        IT| Pune| 45000|         3|   43250.0|
|   104|Sneha| 35|   Finance|Delhi| 60000|         8|   59000.0|
|   106|Pooja| 32|        HR| Pune| 52000|         6|   51000.0|
|   108|Meena| 27|        IT|Delhi| 46000|         3|   43250.0|
+------+-----+---+----------+-----+------+----------+----------+



In [0]:
from pyspark.sql.functions import lag
df.withColumn("prev_sal", lag("salary").over(window)).show()

+------+-----+---+----------+------+------+----------+--------+
|emp_id| name|age|department|  city|salary|experience|prev_sal|
+------+-----+---+----------+------+------+----------+--------+
|   104|Sneha| 35|   Finance| Delhi| 60000|         8|    NULL|
|   107|Karan| 29|   Finance|Mumbai| 58000|         7|   60000|
|   106|Pooja| 32|        HR|  Pune| 52000|         6|    NULL|
|   102| Neha| 30|        HR|Mumbai| 50000|         5|   52000|
|   108|Meena| 27|        IT| Delhi| 46000|         3|    NULL|
|   103|Rahul| 28|        IT|  Pune| 45000|         3|   46000|
|   105|  Raj| 26|        IT|Mumbai| 42000|         2|   45000|
|   101| Amit| 25|        IT|  Pune| 40000|         2|   42000|
+------+-----+---+----------+------+------+----------+--------+



In [0]:
orders_data = [
(1,101,201,2,"2024-01-10","Mumbai"),
(2,102,202,1,"2024-01-11","Pune"),
(3,101,203,3,"2024-01-15","Mumbai"),
(4,103,204,1,"2024-02-01","Delhi"),
(5,104,205,2,"2024-02-05","Mumbai"),
(6,105,201,1,"2024-02-10","Pune"),
(7,102,206,1,"2024-03-01","Pune"),
(8,106,202,2,"2024-03-05","Delhi"),
(9,107,204,1,"2024-03-08","Mumbai"),
(10,108,203,4,"2024-03-10","Pune")
]

orders_columns = [
"order_id","customer_id","product_id",
"quantity","order_date","city"
]

In [0]:
customers_data = [
(101,"Amit",25,"Mumbai"),
(102,"Neha",30,"Pune"),
(103,"Rahul",28,"Delhi"),
(104,"Sneha",35,"Mumbai"),
(105,"Raj",26,"Pune"),
(106,"Pooja",32,"Delhi"),
(107,"Karan",29,"Mumbai"),
(108,"Meena",27,"Pune")
]

customers_columns = [
"customer_id","name","age","city"
]

In [0]:
products_data = [
(201,"Laptop","Electronics",50000),
(202,"Mobile","Electronics",20000),
(203,"Headphones","Electronics",3000),
(204,"Shoes","Fashion",4000),
(205,"T-Shirt","Fashion",1500),
(206,"Watch","Fashion",7000)
]

products_columns = [
"product_id","product_name","category","price"
]

In [0]:
orders_df = spark.createDataFrame(orders_data,orders_columns)
customers_df = spark.createDataFrame(customers_data,customers_columns)
products_df = spark.createDataFrame(products_data,products_columns)

In [0]:
orders_df.join(customers_df,"customer_id").join(products_df, "product_id").select("order_id","name","product_name").show()

+--------+-----+------------+
|order_id| name|product_name|
+--------+-----+------------+
|       1| Amit|      Laptop|
|       2| Neha|      Mobile|
|       3| Amit|  Headphones|
|       4|Rahul|       Shoes|
|       5|Sneha|     T-Shirt|
|       6|  Raj|      Laptop|
|       7| Neha|       Watch|
|       8|Pooja|      Mobile|
|       9|Karan|       Shoes|
|      10|Meena|  Headphones|
+--------+-----+------------+



In [0]:
from pyspark.sql.functions import col
orders_df.join(products_df, "product_id").withColumn("total_sales", col("quantity")*col("price")).show()

+----------+--------+-----------+--------+----------+------+------------+-----------+-----+-----------+
|product_id|order_id|customer_id|quantity|order_date|  city|product_name|   category|price|total_sales|
+----------+--------+-----------+--------+----------+------+------------+-----------+-----+-----------+
|       201|       1|        101|       2|2024-01-10|Mumbai|      Laptop|Electronics|50000|     100000|
|       202|       2|        102|       1|2024-01-11|  Pune|      Mobile|Electronics|20000|      20000|
|       203|       3|        101|       3|2024-01-15|Mumbai|  Headphones|Electronics| 3000|       9000|
|       204|       4|        103|       1|2024-02-01| Delhi|       Shoes|    Fashion| 4000|       4000|
|       205|       5|        104|       2|2024-02-05|Mumbai|     T-Shirt|    Fashion| 1500|       3000|
|       201|       6|        105|       1|2024-02-10|  Pune|      Laptop|Electronics|50000|      50000|
|       206|       7|        102|       1|2024-03-01|  Pune|    

In [0]:
orders_df.join(products_df,"product_id").withColumn("total_sales", col("quantity")* col("price")).groupBy("city").agg(sum("total_Sales").alias("total_revenue")).show()

+------+-------------+
|  city|total_revenue|
+------+-------------+
|Mumbai|       116000|
|  Pune|        89000|
| Delhi|        44000|
+------+-------------+



In [0]:
orders_df.join(products_df,"product_id").withColumn("total_Sales", col("quantity")* col("price")).groupby("category").agg(sum("total_sales").alias("total_revenue")).show()

+-----------+-------------+
|   category|total_revenue|
+-----------+-------------+
|Electronics|       231000|
|    Fashion|        18000|
+-----------+-------------+



In [0]:
customers_df = spark.createDataFrame([
(1,"Rahul","Mumbai","2022-01-10"),
(2,"Sneha","Pune","2022-03-15"),
(3,"Amit","Delhi","2022-05-20"),
(4,"Priya","Mumbai","2023-02-01"),
(5,"Karan","Bangalore","2023-03-18")
],["customer_id","customer_name","city","signup_date"])


products_df = spark.createDataFrame([
(101,"Laptop","Electronics",70000),
(102,"Mobile","Electronics",30000),
(103,"Shoes","Fashion",5000),
(104,"Watch","Fashion",8000),
(105,"Tablet","Electronics",40000)
],["product_id","product_name","category","price"])


orders_df = spark.createDataFrame([
(1,1,101,1,"2023-01-01"),
(2,2,102,2,"2023-01-05"),
(3,1,103,1,"2023-02-10"),
(4,3,101,1,"2023-02-20"),
(5,4,104,2,"2023-03-05"),
(6,5,105,1,"2023-03-10")
],["order_id","customer_id","product_id","quantity","order_date"])

In [0]:
customers_df.filter(customers_df.city == "Mumbai").show()

+-----------+-------------+------+-----------+
|customer_id|customer_name|  city|signup_date|
+-----------+-------------+------+-----------+
|          1|        Rahul|Mumbai| 2022-01-10|
|          4|        Priya|Mumbai| 2023-02-01|
+-----------+-------------+------+-----------+



In [0]:
orders_df.count()

6

In [0]:
products_df.select("product_name").filter(products_df.price > 30000).show()

+------------+
|product_name|
+------------+
|      Laptop|
|      Tablet|
+------------+



In [0]:
customers_df.filter(customers_df.signup_date > "2022-12-31").show()

+-----------+-------------+---------+-----------+
|customer_id|customer_name|     city|signup_date|
+-----------+-------------+---------+-----------+
|          4|        Priya|   Mumbai| 2023-02-01|
|          5|        Karan|Bangalore| 2023-03-18|
+-----------+-------------+---------+-----------+



In [0]:
orders_df.join(products_df,"product_id","inner").withColumn("total_revenue", col("quantity")* col("price")).agg(sum("total_revenue")).show()

+------------------+
|sum(total_revenue)|
+------------------+
|            261000|
+------------------+



In [0]:
orders_df.join(products_df,"product_id").join(customers_df,"customer_id").select("order_id","customer_name","product_name").show()

+--------+-------------+------------+
|order_id|customer_name|product_name|
+--------+-------------+------------+
|       1|        Rahul|      Laptop|
|       2|        Sneha|      Mobile|
|       3|        Rahul|       Shoes|
|       4|         Amit|      Laptop|
|       5|        Priya|       Watch|
|       6|        Karan|      Tablet|
+--------+-------------+------------+



In [0]:
orders_df.join(products_df,"product_id").join(customers_df,"customer_id").withColumn("total_revenue", col("quantity")* col("price")).groupBy("city").agg(sum("total_revenue")).show()

+---------+------------------+
|     city|sum(total_revenue)|
+---------+------------------+
|   Mumbai|             91000|
|     Pune|             60000|
|    Delhi|             70000|
|Bangalore|             40000|
+---------+------------------+



In [0]:
customers_df.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|   Mumbai|    2|
|     Pune|    1|
|    Delhi|    1|
|Bangalore|    1|
+---------+-----+



In [0]:
from pyspark.sql.functions import col, sum

result = orders_df.join(products_df, "product_id") \
.join(customers_df, "customer_id") \
.withColumn("revenue", col("quantity") * col("price")) \
.groupBy("customer_name") \
.agg(sum("revenue").alias("total_spent")) \
.orderBy(col("total_spent").desc()) \
.limit(3)

result.show()


+-------------+-----------+
|customer_name|total_spent|
+-------------+-----------+
|        Rahul|      75000|
|         Amit|      70000|
|        Sneha|      60000|
+-------------+-----------+

